
# 05 - Foundation Model Router

## Goal

Use a Databricks Foundation Model as a function router.

The model will receive:

- the user question
- the approved function catalog
- examples of expected JSON outputs

The model must NOT answer the business question directly.

The model must only return JSON like:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

## Current flow

User question  
↓  
Prompt builder  
↓  
Databricks Foundation Model  
↓  
Structured JSON  
↓  
Validated router output  

## Next step

In the next notebook, the router output will be passed into the FunctionRegistry.

The FunctionRegistry will execute the safe query against the Delta Table.

## Why this matters

This prevents hallucinations because:

- the model does not calculate sales numbers
- the model does not generate SQL
- the model only selects from approved functions
- Python validates the selected function and arguments
- Delta Tables remain the source of truth

In [0]:
import json
import re

FUNCTION_CATALOG_TABLE = "assistant_function_catalog"
MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

print(f"Function catalog table: {FUNCTION_CATALOG_TABLE}")
print(f"Model endpoint: {MODEL_ENDPOINT}")

Function catalog table: assistant_function_catalog
Model endpoint: databricks-meta-llama-3-3-70b-instruct


In [0]:
catalog_df = spark.table(FUNCTION_CATALOG_TABLE)

display(
    catalog_df.select(
        "function_name",
        "description",
        "arguments_schema_json",
        "example_questions_json",
        "expected_json_example"
    )
)

function_name,description,arguments_schema_json,example_questions_json,expected_json_example
get_product_sales_summary,"Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.","{""product_model"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""Accord"", ""Camry"", ""Civic"", ""Explorer"", ""F150"", ""Mustang"", ""RAV4"", ""Silverado""]}, ""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""How many F150 trucks did I sell this month?"", ""Cu\u00e1ntas F150 he vendido este mes?"", ""What revenue did Silverado generate last month?"", ""Dame el resumen de ventas de Mustang este mes""]","{""function_name"": ""get_product_sales_summary"", ""arguments"": {""product_model"": ""F150"", ""date_range"": ""this_month""}}"
get_top_selling_models,Returns the top selling vehicle models by units sold and revenue for a date range.,"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}, ""limit"": {""type"": ""integer"", ""required"": false, ""default"": 5, ""min"": 1, ""max"": 10}}","[""What are my top selling models this month?"", ""Cu\u00e1les son mis modelos m\u00e1s vendidos este mes?"", ""Show me the top 5 models year to date""]","{""function_name"": ""get_top_selling_models"", ""arguments"": {""date_range"": ""this_month"", ""limit"": 5}}"
get_sales_by_rep,Returns units sold and revenue grouped by sales representative for a date range.,"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""Which sales rep sold the most this month?"", ""Qu\u00e9 vendedor vendi\u00f3 m\u00e1s este mes?"", ""Show sales performance by representative year to date""]","{""function_name"": ""get_sales_by_rep"", ""arguments"": {""date_range"": ""this_month""}}"
get_revenue_by_state,Returns units sold and revenue grouped by customer state for a date range.,"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""How much revenue did we generate by state this month?"", ""Cu\u00e1nto vendimos por estado este mes?"", ""Show revenue by state year to date""]","{""function_name"": ""get_revenue_by_state"", ""arguments"": {""date_range"": ""this_month""}}"
get_average_sale_price,"Returns average, minimum, and maximum sale price for a specific vehicle model in a date range.","{""product_model"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""Accord"", ""Camry"", ""Civic"", ""Explorer"", ""F150"", ""Mustang"", ""RAV4"", ""Silverado""]}, ""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""What was the average sale price for Mustang this month?"", ""Cu\u00e1l fue el ticket promedio de Mustang este mes?"", ""Average sale price for F150 year to date""]","{""function_name"": ""get_average_sale_price"", ""arguments"": {""product_model"": ""Mustang"", ""date_range"": ""this_month""}}"
get_dealership_sales_summary,"Returns units sold, revenue, and average sale price grouped by dealership for a date range.","{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""Show dealership sales this month"", ""Dame ventas por dealership este mes"", ""Which dealership generated the most revenue year to date?""]","{""function_name"": ""get_dealership_sales_summary"", ""arguments"": {""date_range"": ""this_month""}}"


In [0]:
# Convert catalog to text for prompting

def build_function_catalog_text(catalog_rows: list) -> str:
    blocks = []

    for row in catalog_rows:
        arguments_schema = json.loads(row["arguments_schema_json"])
        example_questions = json.loads(row["example_questions_json"])
        expected_json = json.loads(row["expected_json_example"])

        block = f"""
Function name:
{row["function_name"]}

Description:
{row["description"]}

Arguments schema:
{json.dumps(arguments_schema, indent=2)}

Example questions:
{json.dumps(example_questions, indent=2, ensure_ascii=False)}

Expected JSON output:
{json.dumps(expected_json, indent=2)}
"""
        blocks.append(block.strip())

    return "\n\n---\n\n".join(blocks)


catalog_rows = catalog_df.collect()
catalog_text = build_function_catalog_text(catalog_rows)

print(catalog_text)

Function name:
get_product_sales_summary

Description:
Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.

Arguments schema:
{
  "product_model": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "Accord",
      "Camry",
      "Civic",
      "Explorer",
      "F150",
      "Mustang",
      "RAV4",
      "Silverado"
    ]
  },
  "date_range": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "this_month",
      "last_month",
      "year_to_date"
    ]
  }
}

Example questions:
[
  "How many F150 trucks did I sell this month?",
  "Cuántas F150 he vendido este mes?",
  "What revenue did Silverado generate last month?",
  "Dame el resumen de ventas de Mustang este mes"
]

Expected JSON output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

---

Function name:
get_top_selling_models

Description:
Retur

In [0]:
# Create promt builder

def build_router_prompt(user_question: str, catalog_text: str) -> str:
    return f"""
You are a function router for Acme Motors.

Your job is to map the user's business question to exactly one allowed function.

Rules:
1. Do not answer the business question directly.
2. Do not calculate sales numbers.
3. Do not generate SQL.
4. Use only the available functions from the function catalog.
5. Return only valid JSON.
6. The JSON must contain:
   - function_name
   - arguments
7. If the question is not supported by the catalog, return:
{{
  "function_name": "unsupported",
  "arguments": {{
    "reason": "Question is outside the supported function catalog."
  }}
}}

Available function catalog:

{catalog_text}

User question:

{user_question}

Return only valid JSON.
""".strip()

In [0]:
# Test prompt with a question

user_question = "Cuántas F150 he vendido este mes?"

router_prompt = build_router_prompt(
    user_question=user_question,
    catalog_text=catalog_text
)

print(router_prompt)

You are a function router for Acme Motors.

Your job is to map the user's business question to exactly one allowed function.

Rules:
1. Do not answer the business question directly.
2. Do not calculate sales numbers.
3. Do not generate SQL.
4. Use only the available functions from the function catalog.
5. Return only valid JSON.
6. The JSON must contain:
   - function_name
   - arguments
7. If the question is not supported by the catalog, return:
{
  "function_name": "unsupported",
  "arguments": {
    "reason": "Question is outside the supported function catalog."
  }
}

Available function catalog:

Function name:
get_product_sales_summary

Description:
Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.

Arguments schema:
{
  "product_model": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "Accord",
      "Camry",
      "Civic",
      "Explorer",
      "F150",
      "Mustang",
      "RAV4",
      "Silvera

In [0]:
# Extract JSON from a response

def extract_json_object(text: str) -> dict:
    if text is None:
        raise ValueError("Model response is empty.")

    cleaned = text.strip()

    # Remove markdown code fences if the model returns ```json ... ```
    cleaned = re.sub(r"^```json\s*", "", cleaned)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    # Try direct JSON parse first
    try:
        return json.loads(cleaned)
    except Exception:
        pass

    # Fallback: extract first JSON-looking object
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)

    if not match:
        raise ValueError(f"No JSON object found in response: {text}")

    return json.loads(match.group(0))

In [0]:
def route_with_ai_query(user_question: str) -> dict:
    prompt = build_router_prompt(
        user_question=user_question,
        catalog_text=catalog_text
    )

    escaped_prompt = prompt.replace("'", "''")

    query = f"""
    SELECT ai_query(
      '{MODEL_ENDPOINT}',
      '{escaped_prompt}'
    ) AS model_response
    """

    response_df = spark.sql(query)
    response_text = response_df.collect()[0]["model_response"]

    return extract_json_object(response_text)

In [0]:
try:
    router_output = route_with_ai_query("Cuántas F150 he vendido este mes?")
    print("Router output from ai_query:")
    print(json.dumps(router_output, indent=2, ensure_ascii=False))
except Exception as error:
    print("ai_query router failed.")
    print("Reason:")
    print(str(error))

Router output from ai_query:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}


In [0]:
def route_with_fallback(user_question: str) -> dict:
    question = user_question.lower()

    # Product summary
    product_models = [
        "f150", "silverado", "mustang", "explorer",
        "camry", "civic", "rav4", "accord"
    ]

    detected_model = None

    for model in product_models:
        if model.lower() in question:
            if model == "f150":
                detected_model = "F150"
            elif model == "rav4":
                detected_model = "RAV4"
            else:
                detected_model = model.capitalize()

    if "last month" in question or "mes pasado" in question:
        date_range = "last_month"
    elif "year to date" in question or "ytd" in question or "año" in question:
        date_range = "year_to_date"
    else:
        date_range = "this_month"

    if detected_model and (
        "cuánt" in question or
        "cuantas" in question or
        "how many" in question or
        "revenue" in question or
        "vendido" in question or
        "vendí" in question or
        "resumen" in question
    ):
        return {
            "function_name": "get_product_sales_summary",
            "arguments": {
                "product_model": detected_model,
                "date_range": date_range
            }
        }

    if (
        "top" in question or
        "más vendidos" in question or
        "mas vendidos" in question or
        "best selling" in question
    ):
        return {
            "function_name": "get_top_selling_models",
            "arguments": {
                "date_range": date_range,
                "limit": 5
            }
        }

    if (
        "vendedor" in question or
        "sales rep" in question or
        "representative" in question
    ):
        return {
            "function_name": "get_sales_by_rep",
            "arguments": {
                "date_range": date_range
            }
        }

    if (
        "estado" in question or
        "state" in question
    ):
        return {
            "function_name": "get_revenue_by_state",
            "arguments": {
                "date_range": date_range
            }
        }

    if (
        "ticket promedio" in question or
        "average sale price" in question or
        "precio promedio" in question
    ) and detected_model:
        return {
            "function_name": "get_average_sale_price",
            "arguments": {
                "product_model": detected_model,
                "date_range": date_range
            }
        }

    if (
        "dealership" in question or
        "dealer" in question or
        "sucursal" in question
    ):
        return {
            "function_name": "get_dealership_sales_summary",
            "arguments": {
                "date_range": date_range
            }
        }

    return {
        "function_name": "unsupported",
        "arguments": {
            "reason": "Question is outside the supported function catalog."
        }
    }

In [0]:
def route_question(user_question: str, prefer_ai_query: bool = True) -> dict:
    if prefer_ai_query:
        try:
            return route_with_ai_query(user_question)
        except Exception as error:
            print("Falling back to local deterministic router.")
            print(f"ai_query error: {error}")

    return route_with_fallback(user_question)

In [0]:
# Test multiple questions

test_questions = [
    "Cuántas F150 he vendido este mes?",
    "Cuáles son mis modelos más vendidos este mes?",
    "Qué vendedor vendió más este mes?",
    "Cuánto vendimos por estado este mes?",
    "Cuál fue el ticket promedio de Mustang este mes?",
    "Dame ventas por dealership este mes",
    "Recomiéndame una laptop gamer"
]

for question in test_questions:
    print("=" * 80)
    print("Question:", question)

    router_output = route_question(question, prefer_ai_query=True)

    print("Router output:")
    print(json.dumps(router_output, indent=2, ensure_ascii=False))

Question: Cuántas F150 he vendido este mes?
Router output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}
Question: Cuáles son mis modelos más vendidos este mes?
Router output:
{
  "function_name": "get_top_selling_models",
  "arguments": {
    "date_range": "this_month",
    "limit": 5
  }
}
Question: Qué vendedor vendió más este mes?
Router output:
{
  "function_name": "get_sales_by_rep",
  "arguments": {
    "date_range": "this_month"
  }
}
Question: Cuánto vendimos por estado este mes?
Router output:
{
  "function_name": "get_revenue_by_state",
  "arguments": {
    "date_range": "this_month"
  }
}
Question: Cuál fue el ticket promedio de Mustang este mes?
Router output:
{
  "function_name": "get_average_sale_price",
  "arguments": {
    "product_model": "Mustang",
    "date_range": "this_month"
  }
}
Question: Dame ventas por dealership este mes
Router output:
{
  "function_name": "get_dealership_

In [0]:
# Validate router output structure

def validate_router_output(router_output: dict) -> dict:
    if not isinstance(router_output, dict):
        return {
            "is_valid": False,
            "error": "Router output must be a dictionary."
        }

    if "function_name" not in router_output:
        return {
            "is_valid": False,
            "error": "Missing function_name."
        }

    if "arguments" not in router_output:
        return {
            "is_valid": False,
            "error": "Missing arguments."
        }

    if not isinstance(router_output["arguments"], dict):
        return {
            "is_valid": False,
            "error": "arguments must be a dictionary."
        }

    return {
        "is_valid": True,
        "error": None
    }

In [0]:
# Test validation

sample_router_output = route_question(
    "Cuántas F150 he vendido este mes?",
    prefer_ai_query=True
)

validation_result = validate_router_output(sample_router_output)

print("Router output:")
print(json.dumps(sample_router_output, indent=2, ensure_ascii=False))

print("\nValidation:")
print(json.dumps(validation_result, indent=2, ensure_ascii=False))

Router output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

Validation:
{
  "is_valid": true,
  "error": null
}



# Notebook result

In this notebook we created a Foundation Model Router.

## What the router does

Input:

Natural language question

Output:

Structured function call JSON

Example:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

## Important

The router does not answer the business question.

The router does not calculate sales numbers.

The router does not generate SQL.

It only chooses a safe function and arguments.

## Flow now

User question  
↓  
Prompt Builder  
↓  
Databricks Foundation Model / ai_query  
↓  
JSON router output  
↓  
Validation  

## Next notebook

Notebook 06 will connect:

Router output  
↓  
FunctionRegistry  
↓  
Safe Delta query  
↓  
Final answer  
↓  
Audit log